# Minimax — Adversário da rede

Neste notebook vamos definir o Minimax, conferir uma jogada e executar os modos médio e difícil. O código das funções é o mesmo de `minimax.py`.

Escolha o kernel da `.venv` e execute as células na ordem. As dependências estão em `requirements-notebook-t2.txt`.

In [ ]:
from pathlib import Path
import sys
import json
from html import escape
import numpy as np
from IPython.display import HTML, display

# O kernel pode começar na pasta do notebook ou na raiz do projeto.
pasta_atual = Path.cwd().resolve()
raiz = next((p for p in [pasta_atual, *pasta_atual.parents] if (p / "app_t2.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Abra o notebook dentro do repositório.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from functools import lru_cache
from t2.jogo import result

In [ ]:
def mostrar_tabela(colunas, linhas):
    cabecalho = "".join(f"<th style='padding:8px;text-align:left'>{escape(str(c))}</th>" for c in colunas)
    corpo = "".join("<tr>" + "".join(f"<td style='padding:8px;border-top:1px solid #ddd'>{escape(str(v))}</td>" for v in linha) + "</tr>" for linha in linhas)
    display(HTML(f"<table><thead><tr>{cabecalho}</tr></thead><tbody>{corpo}</tbody></table>"))

## Minimax completo

A avaliação usa a perspectiva do jogador da vez, na forma Negamax. Quando mais de uma jogada tem o melhor valor, a função retorna todas. Para jogar, o desempate usa o menor índice.

In [ ]:
@lru_cache(maxsize=None)
def minimax(board, player):
    """
    Avalia o tabuleiro do ponto de vista do jogador que está na vez.

    Quando mais de uma ação tem o mesmo melhor valor, o desempate usado
    para jogar é pelo menor índice. Assim a escolha não depende de sorteio.
    """
    end = result(board)
    if end is not None:
        return end * player, ()
    scores = []
    for i, value in enumerate(board):
        if value == 0:
            child = list(board)
            child[i] = player
            score, _ = minimax(tuple(child), -player)
            scores.append((-score, i))
    best = max(s for s, _ in scores)
    return best, tuple(i for s, i in scores if s == best)

## Médio e difícil

O difícil sempre usa Minimax. No médio, a cada turno há 50% de chance de uma jogada aleatória válida. O sorteio acontece antes de chamar o Minimax, sem mudar a busca.

In [ ]:
def opponent_move(board, player, mode, rng):
    available = [i for i,v in enumerate(board) if v == 0]
    if mode == 'medio' and rng.random() < .5:
        return int(rng.choice(available))
    return minimax(tuple(board), player)[1][0]

## Conferir uma posição

No tabuleiro abaixo, é a vez de X. Vamos olhar o valor da posição e as casas que têm a melhor avaliação.

In [ ]:
tabuleiro = [1, 1, 0, -1, -1, 0, 0, 0, 0]
jogador = 1
valor, melhores = minimax(tuple(tabuleiro), jogador)
mostrar_tabela(["Casa", "Ocupação"], [[i+1, v] for i, v in enumerate(tabuleiro)])
print("Valor para quem está na vez:", valor)
print("Melhores casas:", [i+1 for i in melhores])
rng = np.random.default_rng(42)
print("Jogada no difícil:", opponent_move(tabuleiro, jogador, "dificil", rng)+1)

## Sorteios no modo médio

Repetimos a escolha no mesmo tabuleiro para observar o modo médio. Uma jogada aleatória também pode coincidir com uma jogada ótima. Portanto, olhar só a casa escolhida não revela qual caminho do sorteio foi usado.

In [ ]:
rng = np.random.default_rng(42)
mostrar_tabela(["Tentativa", "Casa escolhida no médio"], [
    [i+1, opponent_move(tabuleiro, jogador, "medio", rng)+1] for i in range(20)
])

## Partida completa

Aqui os dois jogadores usam Minimax difícil. O controle de vitória e empate continua sendo feito pelas regras do jogo. Depois podemos mudar o modo de um deles para observar outra partida.

In [ ]:
tabuleiro = [0]*9
jogador = 1
rng = np.random.default_rng(42)
modos = {1: "dificil", -1: "dificil"}
jogadas = []
while result(tabuleiro) is None:
    casa = opponent_move(tabuleiro, jogador, modos[jogador], rng)
    tabuleiro[casa] = jogador
    jogadas.append([len(jogadas)+1, "X" if jogador == 1 else "O", casa+1])
    jogador *= -1
mostrar_tabela(["Jogada", "Jogador", "Casa"], jogadas)
print("Resultado:", {1: "X venceu", -1: "O venceu", 0: "Empate"}[result(tabuleiro)])

## Uso no treinamento

Durante a avaliação, a rede joga contra esse adversário e alternamos quem começa. No médio são necessárias várias partidas por causa da aleatoriedade. No difícil determinístico, usamos uma partida para cada posição inicial.

O ciclo completo fica em [genetico.ipynb](../genetico/genetico.ipynb). Os arquivos `.py` são os usados pelo jogo. Alterações feitas nas células não modificam automaticamente esses arquivos.